<a href="https://colab.research.google.com/github/JiaCheng0427/Problem-Set-1/blob/main/Econ_5200_PS1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 5200 PS1 — The Measurement Audit

**Starter notebook.** The Canvas assignment is the brief: read it first. This
notebook gives you the structure, one heading per step in the order the rubric
grades them, and the module skeleton the brief asks for. Every result is yours
to compute.

**Before you start**
1. *File › Save a copy in Drive*, and rename the copy `Econ 5200 PS1`.
2. Every claim gets a text cell. A deliverable that cannot be read without
   running it is not a deliverable.
3. **AI is not allowed in Phases 1–3.** Phase 4 requires it, with the
   transcript in `ai-appendix.md` in your repository.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw is reproducible

---
## Phase 1: Reproduce the Wrong Number (15 pts)

### Step 1.1: Build the transaction data

> ### A tool you have not used yet — drawing skewed data
>
> `rng.lognormal(mean, sigma, size)` draws right-skewed positive values: most
> are modest, a few are large. `mean` and `sigma` describe the logarithm of the
> values, not the values themselves, so check `np.median(...)` of a draw
> before you settle on them. `rng.choice(...)` and `rng.integers(...)` are
> useful for the ids and the years.

At least 20,000 rows over two years: `customer_id`, `year`, `basket_value`.
Plant the B2B orders and the change in what gets logged, and record the true
consumer average for each year **before** you contaminate anything.

In [ ]:
import numpy as np

rng = np.random.default_rng(5200)

basket_y1 = rng.lognormal(3.91, 0.5, 10000)
basket_y2 = rng.lognormal(3.91, 0.5, 10000)

basket_y1[:10]

array([ 60.85395601, 113.17885118,  26.30572881,  38.94356916,
        35.28789111,  36.48474843,  55.24170148,  58.21747265,
        55.874686  ,  69.45669382])

In [ ]:
# The truth: the consumer average per year, recorded before contamination
true_y1 = basket_y1.mean()
true_y2 = basket_y2.mean()

print("True Year 1 average:", round(true_y1, 2))
print("True Year 2 average:", round(true_y2, 2))

true_change = (true_y2 / true_y1 - 1) * 100
print("True year-over-year change:", round(true_change, 2), "%")

True Year 1 average: 56.79
True Year 2 average: 56.15
True year-over-year change: -1.13 %


In [ ]:
# Contamination 1: B2B orders
b2b_rng = np.random.default_rng(5201)
b2b_y2 = b2b_rng.lognormal(7.0, 0.25, 40)
print("Number of B2B orders:", len(b2b_y2))
print("Average B2B order value:", round(b2b_y2.mean(), 2))

Number of B2B orders: 40
Average B2B order value: 1126.55


In [ ]:
# Contamination 2: what gets logged changes between year 1 and year 2
cancel_rng = np.random.default_rng(5202)

cancelled_y2 = cancel_rng.lognormal(5.75, 0.35, 32)

print("Number of cancelled orders logged in Year 2:", len(cancelled_y2))
print("Average cancelled order value:", round(cancelled_y2.mean(), 2))

Number of cancelled orders logged in Year 2: 32
Average cancelled order value: 356.02


### Step 1.2: Reproduce the dashboard

Compute the naive `mean(basket_value)` by year, as the dashboard does, and
quantify its gap from the truth.

In [ ]:
# The dashboard's number vs the truth
logged_y2 = np.concatenate([basket_y2, b2b_y2, cancelled_y2])

dashboard_y1 = basket_y1.mean()
dashboard_y2 = logged_y2.mean()

dashboard_change = (dashboard_y2 / dashboard_y1 - 1) * 100
gap = dashboard_change - true_change

print("Dashboard Year 1 average:", round(dashboard_y1, 2))
print("Dashboard Year 2 average:", round(dashboard_y2, 2))
print("Dashboard year-over-year change:", round(dashboard_change, 2), "%")
print("True year-over-year change:", round(true_change, 2), "%")
print("Gap:", round(gap, 2), "percentage points")

Dashboard Year 1 average: 56.79
Dashboard Year 2 average: 61.35
Dashboard year-over-year change: 8.03 %
True year-over-year change: -1.13 %
Gap: 9.16 percentage points


---
## Phase 2: Locate the Cause (40 pts)

### Step 2.1: Separate the two mechanisms (25 pts)

Report each contribution as a number.

In [ ]:
# How much of the gap is the skew and the B2B tail?
no_b2b_y2 = np.concatenate([basket_y2, cancelled_y2])

change_no_b2b = (no_b2b_y2.mean() / dashboard_y1 - 1) * 100

b2b_contribution = dashboard_change - change_no_b2b

print("Change without B2B:", round(change_no_b2b, 2), "%")
print("B2B contribution to the gap:", round(b2b_contribution, 2), "percentage points")

Change without B2B: 0.55 %
B2B contribution to the gap: 7.48 percentage points


In [ ]:
# How much is the change in what was logged?
no_cancelled_y2 = np.concatenate([basket_y2, b2b_y2])

change_no_cancelled = (no_cancelled_y2.mean() / dashboard_y1 - 1) * 100

logging_contribution = dashboard_change - change_no_cancelled

print("Change without cancelled orders:", round(change_no_cancelled, 2), "%")
print("Logging contribution to the gap:", round(logging_contribution, 2), "percentage points")

Change without cancelled orders: 6.38 %
Logging contribution to the gap: 1.65 percentage points


| Mechanism | Contribution to the gap | How you isolated it |
|---|---|---|
| Skew and B2B tail |7.48 percentage point |remove B2B orders and computed the native mean |
| Change in what was logged |1.65 percentage points|remove cancelled orders and computed the native mean |

### Step 2.2: Robust alternatives (15 pts)

`stats.trim_mean(x, 0.1)` drops the top and bottom 10% before averaging.

In [ ]:
# Median, trimmed mean, and the mean after your B2B exclusion rule
from scipy import stats

# Median
median_y1 = np.median(basket_y1)
median_y2 = np.median(logged_y2)

# 10% trimmed mean
trimmed_y1 = stats.trim_mean(basket_y1, 0.1)
trimmed_y2 = stats.trim_mean(logged_y2, 0.1)

# B2B exclusion rule: exclude orders above $500
rule_y1 = basket_y1[basket_y1 <= 500].mean()
rule_y2 = logged_y2[logged_y2 <= 500].mean()

print("Median Year 2:", round(median_y2, 2))
print("Trimmed mean Year 2:", round(trimmed_y2, 2))
print("Mean after B2B exclusion:", round(rule_y2, 2))

Median Year 2: 50.03
Trimmed mean Year 2: 52.72
Mean after B2B exclusion: 56.8


| Estimator | Value | The assumption that makes it right | When it would be wrong |
|---|---|---|---|
| Median |50.03|Extreme orders should not affect the typical basket|When the *dashboard* requires the actual average value|
| Trimmed mean |52.72|Extreme values are mostly noise or unusual orders|When large-scale consumer purchases are justified|
|Mean after B2B exclusion |56.80| Orders with an amount exceeding $500 should not be included in the consumer statistics| when the amount of a legitimate consumer order exceeds USD 500|
**Which number should the dashboard show?** Commit to one and defend it.

*Your answer here.* I will use the mean after excluding the B2B orders. Since the dashboard is supposed to display the average consumer shopping basket value, I believe it is more reasonable to continue using the mean rather than switching to the median or removing the mean. Large B2B orders are not actually consumer purchasing behavior, so excluding them will make the data closer to the standard measurement of the dashboard.

---
## Phase 3: Ship the Fix (20 pts)

### Step 3.1: The module

The skeleton below fixes the names the brief asks for. Replace each body,
write the docstrings properly, and keep the type hints. Lab 4's
`OutlierDetector` is a model for the shape: one function per job, the method
chosen by a parameter, and no reliance on a global variable.

In [ ]:
import os
os.makedirs("src", exist_ok=True)

In [ ]:
%%writefile src/basket_metrics.py
"""Functions for auditing basket data and calculating robust means."""

import numpy as np
import pandas as pd
from scipy import stats


def audit_report(df: pd.DataFrame) -> pd.DataFrame:
    """Return one row per column with missingness, dtype, skew, and outlier count."""
    rows = []

    for col in df.columns:
        x = df[col]

        if pd.api.types.is_numeric_dtype(x):
            q1 = x.quantile(0.25)
            q3 = x.quantile(0.75)
            iqr = q3 - q1

            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr

            outlier_count = ((x < lower) | (x > upper)).sum()
            skew = x.skew()
        else:
            outlier_count = 0
            skew = np.nan

        rows.append({
            "column": col,
            "missingness": x.isna().mean(),
            "dtype": str(x.dtype),
            "skew": skew,
            "outlier_count": outlier_count
        })

    return pd.DataFrame(rows)


def robust_mean(x, method: str = "median") -> float:
    """Return a robust center using median, trimmed mean, or B2B exclusion."""
    x = pd.Series(x).dropna()

    if method == "median":
        return float(x.median())

    if method == "trimmed":
        return float(stats.trim_mean(x, 0.1))

    if method == "exclude_b2b":
        return float(x[x <= 500].mean())

    raise ValueError("Unknown method")

if __name__ == "__main__":
    demo = pd.Series([40, 50, 60, 1000])
    print(robust_mean(demo, "exclude_b2b"))

Overwriting src/basket_metrics.py


### Step 3.2: Verification checkpoint

Import the module back and test it on data it has not seen. An `assert` stops
with an error when its condition is False, which is the point: a failing
assertion tells you something.

In [ ]:
import importlib
import sys
import pandas as pd

sys.path.insert(0, "src")
import basket_metrics
importlib.reload(basket_metrics)

symmetric_data = pd.Series([10, 20, 30, 40, 50])

assert abs(
    basket_metrics.robust_mean(symmetric_data, "median")
    - symmetric_data.mean()
) < 1e-8


skewed_data = pd.Series(
    [10, 20, 30, 40, 50, 60, 70, 80, 90, 1000]
)

assert (
    basket_metrics.robust_mean(skewed_data, "trimmed")
    < skewed_data.mean()
)


b2b_data = pd.Series([40, 50, 60, 1000])

assert (
    basket_metrics.robust_mean(b2b_data, "exclude_b2b")
    == 50
)

---
## Phase 4: AI Expansion (25 pts)

AI is now allowed and required. The transcript goes in `ai-appendix.md` in your
repository, linked from here: the prompt as sent, the raw reply including what
you rejected, and one line on every change you made.

**Link to `ai-appendix.md`:**

### Task 4.1: The adversarial review (15 pts)

Run the prompt from the brief, then run the strongest objection down.

[ai-appendix.md](ai-appendix.md)

**The strongest objection, and the evidence that settles it:**

In [ ]:
# The analysis that settles it
# The analysis that settles it

corrected_mean = logged_y2[logged_y2 <= 500].mean()

print("True Year 2 mean:", round(true_y2, 2))
print("Corrected mean:", round(corrected_mean, 2))
print("Difference:", round(corrected_mean - true_y2, 2))

print("Consumer orders above $500:", (basket_y2 > 500).sum())
print("B2B orders at or below $500:", (b2b_y2 <= 500).sum())
print("Cancelled orders still below $500:", (cancelled_y2 <= 500).sum())

True Year 2 mean: 56.15
Corrected mean: 56.8
Difference: 0.65
Consumer orders above $500: 0
B2B orders at or below $500: 0
Cancelled orders still below $500: 26


**What you found, including if it changes your Phase 2 recommendation:**

*Your answer here.* This part of the objection is correct. The threshold of USD500 successfully excluded all B2B orders from this dataset, while not removing any genuine consumer orders. However, there were still 26 cancelled orders below $500, so the corrected average is USD56.80, while the true value should be USD56.15.

### Task 4.2: The board slide (10 pts)

150 words at most: the headline number, the correction, the confidence, and the
one thing you would still want before betting on it.

*Your slide here.* The dashboard shows an 8.03% increase, but the true consumer average actually decreased by 1.13%. The main reason is that large B2B orders and cancelled orders were included in Year 2. After removing orders above $500, the Year 2 average becomes 56.80, which is much closer to the true value of 56.15. I think this correction is reasonable because the very large orders are not normal consumer purchases. However, it is not perfect because some cancelled orders are still included. Before using this rule in a real dashboard, I would want to know whether any real consumer orders can be above USD500.

---
### Before you submit
- [ ] Truth recorded before contamination, and both mechanisms reported as numbers
- [ ] `src/basket_metrics.py` pushed with the notebook, and at least three assertions passing
- [ ] `ai-appendix.md` in the repository and linked above
- [ ] *Runtime › Restart and run all* runs without errors

In [ ]:
%%writefile ai-appendix.md
# AI Appendix

## Prompt sent to AI

[Prep] Act as a partner reviewing an analytics deliverable before it
goes to a client board. You are looking for the objection that would
embarrass us in the room.

[Request] I audited a retailer's "average basket value" metric. I found
the naive mean was inflated by a right-skewed B2B tail and by a change
in what was logged between years. I recommend the dashboard show a
trimmed mean with an explicit B2B exclusion rule. Give me the three
strongest objections to that recommendation, ordered by how likely the
client is to raise them, and for each one tell me what evidence would
settle it.

[Iterate] Be specific to this metric. Do not give me generic data
quality advice.

[Mechanism Check] For each objection, say whether it is a question
about the data, about the statistic, or about the business definition
of the metric — these need different kinds of answer.

[Evaluate] Which single additional analysis would most reduce our
exposure?


## Raw AI reply

The three strongest objections are:

1. **“Why is the B2B exclusion rule valid?”**
   **Type:** Business definition.
   The client may question whether large orders should automatically be treated as B2B. A high-value consumer purchase could be excluded, or a smaller B2B purchase could remain.
   **Evidence that would settle it:** Compare the rule with known order type or customer type. Check how many true consumer orders are excluded and how many B2B orders remain after the cutoff.

2. **“Why use a trimmed mean instead of the normal mean?”**
   **Type:** Statistic.
   A trimmed mean changes the meaning of “average basket value” because it intentionally removes part of the distribution. If large consumer baskets are real purchases, trimming them may understate actual consumer spending.
   **Evidence that would settle it:** Compare the ordinary mean, median, trimmed mean, and mean after B2B exclusion on consumer-only data. If the ordinary mean after correctly removing B2B orders is stable, trimming may not be necessary.

3. **“Does this actually fix the change in what was logged between the two years?”**
   **Type:** Data.
   Removing B2B orders does not automatically correct a logging change. For example, if cancelled orders appear in Year 2 but not Year 1, they can still affect the metric even after B2B orders are removed.
   **Evidence that would settle it:** Recalculate the Year 2 metric after separately removing the transactions caused by the logging change, then compare it with the proposed corrected metric.

**Single additional analysis that would most reduce our exposure:**
Run a sensitivity analysis that separates the two problems: first remove only B2B orders, then remove only the transactions caused by the logging change, and finally remove both. Compare each result with the clean consumer-only average. This would show whether the proposed correction actually recovers the number the dashboard is supposed to measure.


## Changes I made

- I tested the first objection by checking whether any consumer orders were above $500 and whether any B2B orders were at or below $500.
- I also checked how many cancelled orders remained below $500.
- I kept my Phase 2 recommendation, but noted that the $500 rule does not completely fix the logging problem.

Overwriting ai-appendix.md
